## 5.4 Pytorch 模拟线性回归 - Minibatch + DataLoader

#### 1. 这一节我们要解决什么问题？
* 在 6.3 里，我们一次性把全部数据 x, y 喂给模型训练（全量训练 / full-batch）。
* 但真实深度学习通常用的是 mini-batch 训练：
    * 每次只用一小批数据（比如 8、16、32 条样本）更新一次参数
    * 一轮 epoch 把整个数据集分批走完

简单来说，Batch 决定了模型每次学习“吃”多少数据，而 Epoch 决定了模型把所有数据“吃”了几遍。

#### 2, 核心概念

##### 2.1  什么是 Batch / Iteration / Epoch？
* Batch（批）
    * 一次喂给模型的一组样本。
    * 比如 batch_size=10：一次用 10 条样本来算 loss、算梯度、更新参数。
* Iteration（迭代）
    * 模型完成一次参数更新
    * 处理完一个 Batch 的数据并执行一次 optimizer.step()
* Epoch（一轮）
    * 一轮 = 把整个训练集（包含所有的Batch）完整看一遍。

##### 2.2 它们之间的数学关系
`1 个 Epoch 的 Iteration 数量 = 总样本数 ÷ Batch Size`
假设在 MRI 皮肤病预测的训练集中，你一共收集了 1000 张 MRI 图像数据：
* 如果你将 Batch Size 设置为 10。
* 那么模型需要读取 100 次数据，才能把这 1000 张图片全部看完。
* 也就是说，1 个 Epoch = 100 次 Iterations。在这 1 个 Epoch 中，你的模型权重会被更新 100 次。
* 如果你设定总训练轮数为 50 个 Epochs，那么模型最终会把这 1000 张图片反复看 50 遍，总共进行 5000 次参数更新。

##### 2.3 为什么不用 full-batch（一次喂全部数据）？

Full-batch 的缺点：
* 数据大时放不进显存/内存
* 每次更新太慢
* 梯度太“稳定”，有时反而不利于跳出局部最优

Mini-batch 的优点：
* 更快、更省内存
* 训练更常用、更真实
* 梯度有噪声，很多时候更容易找到更好的解

##### 2.4 为什么需要多个 Epoch？
* 就像人类学习一样，只把一本教材看一遍是记不住所有知识点的。
* 模型需要通过多个 Epoch 反复观察数据，不断微调参数，才能真正“收敛”（Loss 降到最低并稳定下来）。

#### 3. DataLoader 是什么？

##### 3.1 核心定义与作用
* 在深度学习模型训练中，DataLoader（数据加载器）是 PyTorch 提供的一个核心数据分发工具。
* 如果说神经网络是一个高效运转的加工厂，那么 DataLoader 就是连接原材料（存储在硬盘上的数据）和加工车间（显卡）的智能传送带。
* 它的主要作用是将庞大、复杂的数据集（Dataset）进行高效的自动化封装，将其转化为一个可迭代的对象。
* 通过这种抽象封装，开发者无需手动编写循环来切分批次或打乱数据，从而将精力集中在模型架构设计和训练逻辑上。

##### 3.2 DataLoader 的三大核心机制
1. 自动化批处理 (Batching)
    * 当处理体积较大的 MRI 图像数据时，直接将数以千计的高分辨率样本完整加载到显存中会瞬间耗尽硬件资源。
    * DataLoader 能够按照预设的规则，自动将数据集切分为固定大小的子集。
    * 这种分批喂入的机制，使得在 3070 Ti 这种移动版 GPU 的有限显存下，依然可以平稳地完成复杂的张量矩阵运算，避免发生 OOM（Out of Memory）错误。
2. 随机打乱 (Shuffling)
    * 为了防止模型在训练过程中记忆数据的固定输入顺序（即产生过拟合），DataLoader 提供了一键开启的数据洗牌功能。
    * 在每一个 Epoch 开始前，它会自动随机重新排列所有样本的索引，确保模型在每一个批次中都能学到更具泛化性的特征。
3. 多进程加速 (Multiprocessing)
    * 图像的读取、解码以及数据增强（如旋转、缩放）通常涉及大量的 CPU 计算。
    * 如果仅依赖单线程，GPU 会经常处于“停机等待数据”的状态。
    * DataLoader 能够启动多个后台工作进程（Workers），在 CPU 上并行预处理下一批数据。
    * 当 GPU 完成当前批次的计算时，新的数据已经准备就绪，从而大幅提升整体训练吞吐量。

##### 3.3 关键参数解析
1. dataset:
    * 它必须是一个实现了 __len__ 和 __getitem__ 方法的 Dataset 对象。
    * 例如在构建皮肤病预测模型时，该对象负责定义如何从硬盘读取单张扫描图片及其对应的诊断标签。
2. batch_size:
    * 每一个批次包含的样本数量。
    * 该数值的设定需要根据显存大小进行平衡测试，常见的取值为 16、32 或 64。
3. shuffle:
    * 布尔值（True/False）
    * 在训练集（Training Set）上通常设置为 True 以增加随机性；
    * 而在验证集（Validation Set）或测试集上通常设置为 False，因为评估过程不需要随机性，固定顺序有助于结果对比。
4. num_workers:
    * 分配给数据加载的子进程数量。
    * 设置为 0 表示仅在主进程中加载数据。
    * 通常可以根据 CPU 的核心数进行合理设置（如 2 或 4）以加速 IO 密集型任务。

##### 3.4 Dataset 是什么
* Dataset 是“数据集对象”📦，你可以把它理解为：
    * 一个能用 dataset[i] 取出第 i 条样本的东西
* PyTorch 的 DataLoader 需要 Dataset 才能工作。
* 最简单的 Dataset 就是 TensorDataset：
    * 把 x 和 y 绑在一起
    * 每次取样返回 (x_i, y_i)

##### 3.5 与 Dataset 的协同工作模式
* Dataset 负责“个体”：
    * 它像是一本字典，只关心“一共有多少条数据”以及“如何获取第 i 条数据”。
* DataLoader 负责“统筹”：
    * 它拿着这本字典，负责决定“这次抽出哪几页”、“由谁去复印这几页数据”，
    * 最后打包成一个整齐的张量（Tensor）批次交付给模型。


#### 4. 用同一个数据集构建 DataLoader 🧪

In [1]:
import torch
import torch.nn as nn

# 1. 构建数据
x = torch.arange(1,51, dtype=torch.float32)
true_w = 3.0
true_b = 2.0
noise = torch.randn_like(x) * 0.5
y = true_w * x + true_b + noise
x = x.reshape(-1,1)
y = y.reshape(-1,1)

# 2. 定义模型
model = nn.Linear(in_features=1, out_features=1)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

# ✅ 3. 定义DataSet 和 DataLoader
from torch.utils.data import TensorDataset, DataLoader
dataset = TensorDataset(x, y)
batch_size = 4
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

# 4. 训练模型
# ⚠️注意：此时模型训练分为2个循环：外层循环是epoch，内层循环是dataloader
epochs = 30
for epoch in range(1, epochs + 1):
    for batch_x, batch_y in dataloader:
        # 4.1 forward pass
        y_pred = model(batch_x)
        # 4.2 loss
        loss = criterion(y_pred, batch_y)
        # 4.3 backward pass
        loss.backward()
        # 4.4 update weights
        optimizer.step()
        # 4.5 zero gradients
        optimizer.zero_grad()
    # 5. 每一次epoch结束后打印损失
    print(f"Epoch {epoch}/{epochs}, Loss: {loss.item():.4f}")


Epoch 1/30, Loss: 1.6443
Epoch 2/30, Loss: 0.3446
Epoch 3/30, Loss: 4.2250
Epoch 4/30, Loss: 1.4870
Epoch 5/30, Loss: 3.6067
Epoch 6/30, Loss: 0.2449
Epoch 7/30, Loss: 1.1631
Epoch 8/30, Loss: 3.7126
Epoch 9/30, Loss: 10.7950
Epoch 10/30, Loss: 0.2218
Epoch 11/30, Loss: 3.1417
Epoch 12/30, Loss: 4.5138
Epoch 13/30, Loss: 2.0745
Epoch 14/30, Loss: 2.1454
Epoch 15/30, Loss: 3.9599
Epoch 16/30, Loss: 1.7438
Epoch 17/30, Loss: 1.1384
Epoch 18/30, Loss: 0.8011
Epoch 19/30, Loss: 2.4676
Epoch 20/30, Loss: 2.4222
Epoch 21/30, Loss: 1.9670
Epoch 22/30, Loss: 0.8142
Epoch 23/30, Loss: 12.4118
Epoch 24/30, Loss: 1.3835
Epoch 25/30, Loss: 1.0468
Epoch 26/30, Loss: 1.4558
Epoch 27/30, Loss: 0.6161
Epoch 28/30, Loss: 32.6290
Epoch 29/30, Loss: 0.4283
Epoch 30/30, Loss: 0.2844


#### 5. 流程重点讲解

##### 5.1 为什么有两层循环？
* 外层：epoch
    * 控制“数据集重复训练多少次”
    * 每个 epoch = 完整遍历一次 dataset
* 内层：batch
    * 每次取一个 batch（由 DataLoader 自动给）
    * 每个 batch 更新一次参数

##### 5.2 为什么 zero_grad() 要放在 batch 循环里？

因为你每个 batch 都要：
* backward 产生新梯度
* 如果不清零，会发生梯度累加（错误）

##### 5.3 为什么 shuffle=True 很重要？

如果你不 shuffle：
* 数据顺序固定
* 每个 batch 可能都有规律（例如先小 x 再大 x）
* 可能导致训练不稳定或收敛慢

shuffle=True：
* 每个 epoch 都重新打乱顺序
* batch 更像“真实随机采样”


##### 5.4 为什么 loss 在 mini-batch 下会波动？
* 每个 batch 数据不同
* 每个 batch 的 loss 不同
* 更新方向会有噪声

这不是 bug，是 mini-batch 的正常现象 ✅

如果你想看“更稳定”的 loss：
* 记录每个 batch 的 loss
* 然后在 epoch 末做平均

#### 6. 七、改进：打印每个 epoch 的平均 loss（推荐）📊
```
for epoch in range(1, epochs + 1):
    epoch_loss = 0.0

    for batch_x, batch_y in dataloader:
        y_pred = model(batch_x)
        loss = criterion(y_pred, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)

    if epoch % 10 == 0:
        w = model.weight.item()
        b = model.bias.item()
        print(f"epoch {epoch:03d} | avg_loss={avg_loss:.6f} | w={w:.4f} | b={b:.4f}")
```